In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch01-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "01-01-molecular-graphs.py"

# 乙醇和二甲醚的氧各連著誰？

改動下一格的 Python 腳本後，請重新執行該格與最後一格。ESOL 首次載入需要下載，這裡只讀取結構。

In [ ]:
%%writefile "{script}"
# 概念示意 — 實際應用請使用下方 DeepChem 版本
from pathlib import Path
import os
from rdkit import Chem
from rdkit.Chem import Draw, rdMolDescriptors

def oxygen_neighbors(mol):
    oxygen = next(atom for atom in mol.GetAtoms() if atom.GetSymbol() == "O")
    neighbors = [atom.GetSymbol() for atom in oxygen.GetNeighbors()]
    return oxygen.GetIdx(), neighbors, oxygen.GetTotalNumHs()

examples = [("Ethanol", "CCO"), ("Dimethyl ether", "COC")]
mols = [Chem.MolFromSmiles(smiles) for _, smiles in examples]
for (name, _), mol in zip(examples, mols):
    print(name, rdMolDescriptors.CalcMolFormula(mol), oxygen_neighbors(mol))
    for bond in mol.GetBonds():
        print(" ", bond.GetBeginAtomIdx(), bond.GetEndAtomIdx(), bond.GetBondType())

options = Draw.MolDrawOptions()
options.addAtomIndices = True
options.explicitMethyl = True
options.legendFontSize = 24
panel = Draw.MolsToGridImage(mols, molsPerRow=2, subImgSize=(360, 230),
                             legends=[name for name, _ in examples], drawOptions=options)
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)
panel.save(out / "01-01-molecular-graphs.png")

# DeepChem 版本：保留 ESOL 資料列，逐列算分子式，再讀取氧的鄰居。
import deepchem as dc

_, parts, _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter="index",
    transformers=[], reload=False, data_dir=os.environ.get("DEEPCHEM_DATA_DIR"))
for part in parts:
    for smiles in part.ids:
        mol = Chem.MolFromSmiles(smiles)
        if rdMolDescriptors.CalcMolFormula(mol) == "C2H6O":
            print("ESOL", smiles, oxygen_neighbors(mol))

In [ ]:
from IPython.display import Image, display

run = subprocess.run([python, str(script)], capture_output=True, text=True)
if run.returncode:
    print(run.stderr)
    run.check_returncode()
print(run.stdout.replace("This module requires PyTorch to be installed.\n", "").strip())
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "01-01-molecular-graphs.png")))